# 06 · PrimateAI — the semi-supervised alternative

**PrimateAI** (Sundaram et al. 2018, *Nat Genet*, PMID 30038395) is a deep net that dodges the need for curated pathogenic labels: it trains on a **proxy for benignity** — missense variants common in humans or seen in other primates. That makes it **semi-supervised**, with only **medium** circularity vs ClinVar.

> ⚠️ **DEMO DATA.** The PrimateAI scores here are hand-authored illustrative values for ~13 curated CFTR variants — **not** real PrimateAI output. Every table keeps a `source` column reading `DEMO`. See the *How to get REAL data* box, then join real per-variant scores on `protein_variant`; the code runs unchanged once `source` is `REAL`.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
%matplotlib inline

## 5 · PrimateAI — the *semi-supervised* alternative

**PrimateAI** (Sundaram et al. 2018, *Nature Genetics*, **PMID 30038395**) is a **deep neural
network**, but it dodges the "we need labelled pathogenic variants" problem with a neat trick.

Instead of training on curated *pathogenic* labels, it trains on a **proxy for benignity**:

> A missense variant that is **common** in healthy humans — or is seen as the normal amino acid in
> **other primates** (chimp, gorilla, …) — is very likely **tolerated / benign**. Evolution has
> already "tested" it.

So PrimateAI learns from **hundreds of thousands of common human & non-human-primate missense
variants** as its benign class. This is why it's called **semi-supervised**: it uses labels, but
those labels are a *population-frequency proxy*, **not** clinical pathogenic/benign assertions.

| Property | PrimateAI |
|---|---|
| Learning type | **Semi-supervised** (benign proxy from primate/common variants) |
| Score range | **0 → 1** (higher = more likely pathogenic) |
| Pathogenic cut | **≥ 0.803** |
| Circularity vs ClinVar | **Medium** — it never trained on ClinVar *pathogenic* labels, but common-variant proxies can still overlap benign ClinVar entries |


In [2]:
tk.THRESHOLDS['primate_ai']

{'path': 0.803, 'benign': 0.483}

## 2 · Load PrimateAI and make a call

Score 0-1, higher = worse, pathogenic cut `>= 0.803`.

In [3]:
primateai = tk.load_primateai()
primateai['pai_call'] = primateai['primate_ai_score'].apply(lambda s: tk.call_from_score(s, 'primate_ai'))
primateai[['protein_variant', 'primate_ai_score', 'pai_call', 'source']]

,protein_variant,primate_ai_score,pai_call,source
0,G551D,0.930,pathogenic,DEMO
2,R117H,0.440,benign,DEMO
3,R334W,0.850,pathogenic,DEMO
4,G85E,0.880,pathogenic,DEMO
5,D1152H,0.600,uncertain,DEMO
6,R668C,0.280,benign,DEMO
7,Tyr161Cys,0.841,pathogenic,DEMO
8,Gly970Asp,0.782,uncertain,DEMO
9,Ser912Leu,0.751,uncertain,DEMO
10,Val520Phe,0.731,uncertain,DEMO


## 3 · How to get the REAL PrimateAI scores

PrimateAI / PrimateAI-3D scores are distributed by **Illumina** (accept a data-use agreement, then download the precomputed files). Keyed by **genomic coordinate** — join on `(chr, pos, ref, alt)`, minding CFTR's minus strand.

In [4]:
info = tk.TOOL_REGISTRY['PrimateAI']
for key, val in info.items():
    print(f'  {key:12s}: {val}')

  kind        : missense
  learning    : semi-supervised
  signal      : deep net trained on common human/primate missense as a benign proxy
  circularity : medium
  pmid        : 30038395


## Key takeaways

1. **PrimateAI** is **semi-supervised**: benignity learned from common human/primate variants, not clinical labels → **medium** circularity.
2. Score 0-1, cut `>= 0.803`.
3. **DEMO** data — real scores from Illumina, joined on genomic coordinate.

**Next:** notebook 07 — **ClinVar**, the first of the two clinical truth sets.